# autoresearch — A4000 Cloud Notebook Setup

Karpathy's autoresearch: AI agent autonomously runs 5-min training experiments, keeps improvements, discards failures, and loops indefinitely.

**Cloud kernel: RTX A4000 (16 GB VRAM), PyTorch 2.1.1+cu121, Python 3.11.** Uses `pip` (no uv on this kernel).

**Steps:** 1. GPU check → 2. Install deps → 3. A4000 patches → 4. Research libraries → 5. Prepare data → 6. Baseline run → 7. Agent loop

## 1. GPU Check

In [3]:
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                    "--format=csv,noheader"], capture_output=True, text=True)
print(r.stdout)
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    cap = torch.cuda.get_device_capability()
    arch = "Hopper" if cap == (9,0) else "Ampere/Ada" if cap[0] == 8 else "Other"
    print(f"Cap:     {cap} ({arch})")

NVIDIA RTX A4000, 16376 MiB, 8.6

PyTorch: 2.1.1+cu121
CUDA:    True
GPU:     NVIDIA RTX A4000
VRAM:    15.7 GB
Cap:     (8, 6) (Ampere/Ada)


## 2. Install Core Dependencies

Cloud kernel uses `pip` directly. Torch 2.1.1+cu121 is already present — skip reinstalling it unless you need a specific version.

In [4]:
import sys, subprocess

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args, "-q"])
    return r.returncode == 0

# Core project deps (skip torch — already installed on cloud kernel)
core = [
    "kernels",       # FA3 / GPU kernel loader
    "matplotlib",    # plotting
    "numpy",
    "pandas",
    "pyarrow",       # parquet reading
    "requests",
    "rustbpe",       # fast BPE tokenizer (Rust)
    "tiktoken",      # tokenizer utils
]
for pkg in core:
    ok = pip(pkg)
    print(f"  {'OK' if ok else 'FAIL':4s}  {pkg}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradient 2.0.6 requires attrs<=19, but you have attrs 23.1.0 which is incompatible.
gradient 2.0.6 requires PyYAML==5.*, but you have pyyaml 6.0.3 which is incompatible.
transformers 4.35.2 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.6.0 which is incompatible.
tokenizers 0.15.1 requires huggingface_hub<1.0,>=0.16.4, but you have huggingface-hub 1.6.0 which is incompatible.
datasets 2.14.5 requires huggingface-hub<1.0.0,>=0.14.0, but you have huggingface-hub 1.6.0 which is incompatible.


  OK    kernels


  OK    matplotlib


  OK    numpy


  OK    pandas


  OK    pyarrow


  OK    requests


  OK    rustbpe
  OK    tiktoken


## 3. A4000 Patches

### 3a. Flash Attention 3 check

FA3 targets H100 (sm_90). On A4000 (sm_86) it uses `kernels-community/flash-attn3`.
Test below — if it fails run **3b** to swap in PyTorch SDPA.

In [6]:
import torch
cap = torch.cuda.get_device_capability()
repo = "varunneal/flash-attention-3" if cap == (9, 0) else "kernels-community/flash-attn3"
print(f"Testing: {repo}")
try:
    from kernels import get_kernel
    fa3 = get_kernel(repo).flash_attn_interface
    q = torch.randn(2, 16, 4, 32, dtype=torch.bfloat16, device="cuda")
    k, v = torch.randn_like(q), torch.randn_like(q)
    out = fa3.flash_attn_func(q, k, v, causal=True)
    print(f"FA3 OK: output shape {out.shape}")
except Exception as e:
    print(f"FA3 FAILED: {e}")
    print("-> Run cell 3b to patch with PyTorch SDPA fallback")

Testing: kernels-community/flash-attn3


Fetching 0 files: 0it [00:00, ?it/s]

FA3 FAILED: Cannot install kernel from repo kernels-community/flash-attn3 (revision: main)
-> Run cell 3b to patch with PyTorch SDPA fallback


In [18]:
# 3b. Patch train.py: swap FA3 for PyTorch SDPA  (only run if 3a failed)
with open("train.py", "r") as f:
    src = f.read()

if "# [A4000-PATCHED]" in src:
    print("Already patched.")
else:
    old = (
        "from kernels import get_kernel\n"
        "cap = torch.cuda.get_device_capability()\n"
        "# varunneal's FA3 is Hopper only, use kernels-community on non-Hopper GPUs\n"
        'repo = "varunneal/flash-attention-3" if cap == (9, 0) else "kernels-community/flash-attn3"\n'
        "fa3 = get_kernel(repo).flash_attn_interface"
    )
    new = (
        "# [A4000-PATCHED] PyTorch SDPA replaces FA3 (FA3 unreliable on Ampere)\n"
        "import torch.nn.functional as _Fsdpa\n"
        "class _FA3:\n"
        "    @staticmethod\n"
        "    def flash_attn_func(q, k, v, causal=True, window_size=(-1, 0)):\n"
        "        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)\n"
        "        ws = window_size[0] if isinstance(window_size, (tuple, list)) else window_size\n"
        "        if ws > 0 and ws < q.shape[2]:\n"
        "            T = q.shape[2]\n"
        "            m = torch.ones(T,T,dtype=torch.bool,device=q.device).tril()\n"
        "            m = m & torch.ones(T,T,dtype=torch.bool,device=q.device).triu(-(ws-1))\n"
        "            am = torch.zeros(T,T,dtype=q.dtype,device=q.device).masked_fill_(~m, float('-inf'))\n"
        "            out = _Fsdpa.scaled_dot_product_attention(q, k, v, attn_mask=am)\n"
        "        else:\n"
        "            out = _Fsdpa.scaled_dot_product_attention(q, k, v, is_causal=causal)\n"
        "        return out.transpose(1, 2)\n"
        "class _FA3Iface:\n"
        "    flash_attn_func = staticmethod(_FA3.flash_attn_func)\n"
        "fa3 = _FA3Iface()"
    )
    if old in src:
        src = src.replace(old, new)
        open("train.py", "w").write(src)
        print("Patched: FA3 -> PyTorch SDPA")
    else:
        print("WARNING: FA3 block not found — check train.py manually")

Already patched.


### 3c. Tune hyperparameters for 16 GB VRAM

| File | Parameter | H100 default | A4000 | Reason |
|------|-----------|-------------|-------|--------|
| `prepare.py` | `MAX_SEQ_LEN` | 2048 | **1024** | Halves activation memory |
| `prepare.py` | `EVAL_TOKENS` | 40x | **10x** | Faster eval |
| `train.py` | `DEVICE_BATCH_SIZE` | 128 | **16** | Primary VRAM fix |
| `train.py` | `TOTAL_BATCH_SIZE` | 2^19 | **2^17** | grad_accum = 131072/(16×1024) = 8 ✓ |
| `train.py` | `WINDOW_PATTERN` | SSSL | **L** | Skip banded attention overhead |
| `train.py` | `DEPTH` | 8 | **6** | Smaller model |

In [8]:
with open("prepare.py", "r") as f:
    src = f.read()
for old, new in [
    ("MAX_SEQ_LEN = 2048", "MAX_SEQ_LEN = 1024"),
    ("EVAL_TOKENS = 40 * 524288", "EVAL_TOKENS = 10 * 524288"),
]:
    if old in src: src = src.replace(old, new); print(f"prepare.py: {old!r} -> {new!r}")
    elif new in src: print(f"prepare.py: already set: {new!r}")
    else: print(f"WARNING: not found: {old!r}")
open("prepare.py", "w").write(src)
print("prepare.py done")

prepare.py: 'MAX_SEQ_LEN = 2048' -> 'MAX_SEQ_LEN = 1024'
prepare.py: 'EVAL_TOKENS = 40 * 524288' -> 'EVAL_TOKENS = 10 * 524288'
prepare.py done


In [9]:
with open("train.py", "r") as f:
    src = f.read()
for old, new in [
    ("TOTAL_BATCH_SIZE = 2**19", "TOTAL_BATCH_SIZE = 2**17"),
    ("DEVICE_BATCH_SIZE = 128",  "DEVICE_BATCH_SIZE = 16"),
    ('WINDOW_PATTERN = "SSSL"',  'WINDOW_PATTERN = "L"'),
    ("DEPTH = 8",                "DEPTH = 6"),
]:
    if old in src: src = src.replace(old, new); print(f"train.py: {old!r} -> {new!r}")
    elif new in src: print(f"train.py: already set: {new!r}")
    else: print(f"WARNING: not found: {old!r}")
open("train.py", "w").write(src)
import re
tbs = eval(re.search(r'TOTAL_BATCH_SIZE = (.+)', src).group(1))
dbs = int(re.search(r'DEVICE_BATCH_SIZE = (\d+)', src).group(1))
seq = int(re.search(r'MAX_SEQ_LEN = (\d+)', open('prepare.py').read()).group(1))
g = tbs // (dbs * seq)
ok = tbs % (dbs * seq) == 0
print(f'\nGrad accum: {tbs}/({dbs}*{seq}) = {g}  {"OK" if ok else "MISMATCH"}')

train.py: 'TOTAL_BATCH_SIZE = 2**19' -> 'TOTAL_BATCH_SIZE = 2**17'
train.py: 'DEVICE_BATCH_SIZE = 128' -> 'DEVICE_BATCH_SIZE = 16'
train.py: 'WINDOW_PATTERN = "SSSL"' -> 'WINDOW_PATTERN = "L"'
train.py: 'DEPTH = 8' -> 'DEPTH = 6'

Grad accum: 131072/(16*1024) = 8  OK


## 4. Embedding Research Libraries

Extra packages for researching resource-efficient embedding methods inside the autoresearch loop.

| Library | Purpose |
|---------|--------|
| `einops` | Clean tensor reshape/rearrange — essential for factorized & compositional embeddings |
| `bitsandbytes` | int8 / int4 quantized embeddings and linear layers (Ampere supported) |
| `vector-quantize-pytorch` | VQ-VAE / product quantization codebook embeddings |
| `triton` | Write custom CUDA kernels inline (ships with PyTorch, listed for clarity) |
| `scipy` | Cosine similarity, clustering, embedding analysis utils |
| `scikit-learn` | PCA, k-means — useful for embedding space analysis |
| `rich` | Pretty experiment tables in notebook output |
| `safetensors` | Fast tensor save/load for embedding checkpoints |

In [10]:
import sys, subprocess

research_libs = [
    # Embedding manipulation & factorization
    ("einops",                   "tensor reshape for factorized/compositional embeddings"),

    # Quantized embeddings
    ("bitsandbytes",             "int8/int4 quantized embeddings (Ampere sm_86 supported)"),

    # Codebook / product quantization
    ("vector-quantize-pytorch",  "VQ / residual-VQ / product-quantization codebooks"),

    # Custom CUDA kernels
    ("triton",                   "write inline GPU kernels for custom embedding ops"),

    # Analysis
    ("scipy",                    "cosine sim, clustering, embedding geometry analysis"),
    ("scikit-learn",             "PCA/k-means on embedding spaces"),

    # Utilities
    ("rich",                     "pretty tables for experiment results in notebook"),
    ("safetensors",              "fast save/load for embedding checkpoints"),
]

print(f"{'Library':<35} {'Status':<8} Purpose")
print("-" * 80)
for pkg, purpose in research_libs:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True)
    status = "OK" if r.returncode == 0 else "FAIL"
    print(f"{pkg:<35} {status:<8} {purpose}")
print("\nAll done.")

Library                             Status   Purpose
--------------------------------------------------------------------------------
einops                              OK       tensor reshape for factorized/compositional embeddings
bitsandbytes                        OK       int8/int4 quantized embeddings (Ampere sm_86 supported)
vector-quantize-pytorch             OK       VQ / residual-VQ / product-quantization codebooks
triton                              OK       write inline GPU kernels for custom embedding ops
scipy                               OK       cosine sim, clustering, embedding geometry analysis
scikit-learn                        OK       PCA/k-means on embedding spaces
rich                                OK       pretty tables for experiment results in notebook
safetensors                         OK       fast save/load for embedding checkpoints

All done.


In [11]:
# Quick import check — verify all research libs are importable
import importlib, sys

checks = [
    ("einops",           "einops"),
    ("bitsandbytes",     "bitsandbytes"),
    ("vector_quantize_pytorch", "vector_quantize_pytorch"),
    ("triton",           "triton"),
    ("scipy",            "scipy"),
    ("sklearn",          "scikit-learn"),
    ("rich",             "rich"),
    ("safetensors",      "safetensors"),
    ("torch",            "torch"),
    ("tiktoken",         "tiktoken"),
]
for import_name, pkg_name in checks:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, '__version__', 'ok')
        print(f"  OK   {import_name:<30} {ver}")
    except ImportError as e:
        print(f"  FAIL {import_name:<30} {e}")

  OK   einops                         0.8.2
  FAIL bitsandbytes                   No module named 'triton.ops'
  OK   vector_quantize_pytorch        ok
  OK   triton                         3.6.0
  OK   scipy                          1.11.2
  OK   sklearn                        1.3.0
  OK   rich                           ok
  OK   safetensors                    0.4.0
  OK   torch                          2.1.1+cu121
  OK   tiktoken                       0.12.0


## 5. Prepare Data

One-time download (10 training shards + 1 pinned val shard) + BPE tokenizer training.
~2–5 min. Cached at `~/.cache/autoresearch/`.

In [12]:
import os, subprocess, sys
cache = os.path.expanduser("~/.cache/autoresearch")
tok = os.path.join(cache, "tokenizer", "tokenizer.pkl")
data = os.path.join(cache, "data")
if os.path.exists(tok):
    shards = len([f for f in os.listdir(data) if f.endswith(".parquet")]) if os.path.exists(data) else 0
    print(f"Already prepared: {shards} shards, tokenizer exists")
else:
    print("Running prepare.py --num-shards 10 ...")
    r = subprocess.run([sys.executable, "prepare.py", "--num-shards", "10"])
    print("Done!" if r.returncode == 0 else f"Error: returncode={r.returncode}")

Running prepare.py --num-shards 10 ...
Cache directory: /root/.cache/autoresearch

Data: downloading 11 shards (0 already exist)...
  Downloaded shard_00002.parquet
  Downloaded shard_00006.parquet
  Downloaded shard_00004.parquet
  Downloaded shard_00000.parquet
  Downloaded shard_00001.parquet
  Downloaded shard_06542.parquet
  Downloaded shard_00003.parquet
  Downloaded shard_00008.parquet
  Downloaded shard_00007.parquet
  Downloaded shard_00009.parquet
  Downloaded shard_00005.parquet
Data: 11/11 shards ready at /root/.cache/autoresearch/data

Tokenizer: training BPE tokenizer...
Tokenizer: trained in 39.7s, saved to /root/.cache/autoresearch/tokenizer/tokenizer.pkl
Tokenizer: building token_bytes lookup...
Tokenizer: saved token_bytes to /root/.cache/autoresearch/tokenizer/token_bytes.pt
Tokenizer: sanity check passed (vocab_size=8192)

Done! Ready to train.
Done!


## 6. Baseline Training Run

5-minute run on unmodified `train.py`. Sets the `val_bpb` baseline all experiments must beat.

In [13]:
import subprocess, sys, time
print("Starting baseline run (~5 min + startup)... output -> run.log")
t0 = time.time()
r = subprocess.run(
    [sys.executable, "train.py"],
    stdout=open("run.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Finished in {time.time()-t0:.0f}s  (returncode={r.returncode})")

Starting baseline run (~5 min + startup)... output -> run.log
Finished in 351s  (returncode=0)


In [14]:
import subprocess
r = subprocess.run(
    ["grep", "-E",
     "^val_bpb:|^peak_vram_mb:|^num_params_M:|^depth:|^total_tokens_M:|^mfu_percent:",
     "run.log"],
    capture_output=True, text=True
)
if r.stdout.strip():
    print("=== BASELINE RESULTS ===")
    print(r.stdout)
else:
    print("No results — run may have crashed. Last 40 lines of run.log:")
    print("".join(open("run.log").readlines()[-40:]))

=== BASELINE RESULTS ===
val_bpb:          1.185392
peak_vram_mb:     1993.4
mfu_percent:      3.25
total_tokens_M:   88.5
num_params_M:     26.3
depth:            6



In [15]:
# Full log tail (last 3000 chars)
print(open("run.log").read()[-3000:])

oss: 3.321643 | lrm: 0.07 | dt: 453ms | tok/sec: 289,169 | mfu: 3.2% | epoch: 1 | remaining: 10s    
step 00653 (96.7%) | loss: 3.327270 | lrm: 0.07 | dt: 455ms | tok/sec: 288,351 | mfu: 3.2% | epoch: 1 | remaining: 9s    
step 00654 (96.9%) | loss: 3.338005 | lrm: 0.06 | dt: 454ms | tok/sec: 288,992 | mfu: 3.2% | epoch: 1 | remaining: 9s    
step 00655 (97.0%) | loss: 3.331197 | lrm: 0.06 | dt: 453ms | tok/sec: 289,141 | mfu: 3.2% | epoch: 1 | remaining: 8s    
step 00656 (97.2%) | loss: 3.340336 | lrm: 0.06 | dt: 454ms | tok/sec: 288,933 | mfu: 3.2% | epoch: 1 | remaining: 8s    
step 00657 (97.3%) | loss: 3.345188 | lrm: 0.05 | dt: 454ms | tok/sec: 288,995 | mfu: 3.2% | epoch: 1 | remaining: 8s    
step 00658 (97.5%) | loss: 3.331093 | lrm: 0.05 | dt: 454ms | tok/sec: 288,943 | mfu: 3.2% | epoch: 1 | remaining: 7s    
step 00659 (97.6%) | loss: 3.327073 | lrm: 0.05 | dt: 454ms | tok/sec: 288,978 | mfu: 3.2% | epoch: 1 | remaining: 7s    
step 00660 (97.8%) | loss: 3.336656 | lrm: 0.

## 7. Agent Loop

With baseline done, tell Claude Code:
```
Have a look at program.md and let's kick off a new experiment! Let's do the setup first.
```

The agent will: create branch → record results.tsv → loop: edit `train.py` → run → keep/revert → repeat.

Or run experiments manually with the helper below.

In [19]:
# Manual experiment runner
# Edit train.py first, then call run_experiment('my_idea')
import subprocess, sys, time

def run_experiment(label="exp"):
    log = f"run_{label}.log"
    print(f"Running: {label} -> {log}")
    t0 = time.time()
    subprocess.run([sys.executable, "train.py"],
                   stdout=open(log, "w"), stderr=subprocess.STDOUT)
    elapsed = time.time() - t0
    g = subprocess.run(["grep", "-E", "^val_bpb:|^peak_vram_mb:", log],
                       capture_output=True, text=True)
    if not g.stdout.strip():
        print(f"CRASH after {elapsed:.0f}s. Last lines:")
        print("".join(open(log).readlines()[-20:]))
        return None
    m = {}
    for line in g.stdout.strip().split("\n"):
        k, v = line.split(":", 1)
        m[k.strip()] = float(v.strip())
    bpb = m.get("val_bpb", 0)
    vram = m.get("peak_vram_mb", 0) / 1024
    print(f"val_bpb={bpb:.6f} | vram={vram:.1f} GB | {elapsed:.0f}s")
    return bpb, vram

# run_experiment('baseline')

In [20]:
# View results.tsv  (agent updates this after each run)
import os
if os.path.exists("results.tsv"):
    print(open("results.tsv").read())
else:
    print("results.tsv not yet created.")
    # Initialize:
    # open('results.tsv','w').write('commit\tval_bpb\tmemory_gb\tstatus\tdescription\n')

results.tsv not yet created.


In [22]:
import os
cache = os.path.expanduser("~/.cache/autoresearch")
print(os.path.exists(cache))

True
